<p style="align: center;"><img align=center src="https://drive.google.com/uc?export=view&id=1I8kDikouqpH4hf7JBiSYAeNT2IO52T-T" width=600 height=480/></p>
<h3 style="text-align: center;"><b>Школа глубокого обучения ФПМИ МФТИ</b></h3>

<h3 style="text-align: center;"><b>Домашнее задание. Generative adversarial networks</b></h3>



В этом домашнем задании вы обучите GAN генерировать лица людей и посмотрите на то, как можно оценивать качество генерации

In [ ]:
import os
from torch.utils.data import DataLoader
from torchvision.datasets import ImageFolder
import torchvision.transforms as tt
import torch
import torch.nn as nn
import cv2
from tqdm.notebook import tqdm
from torchvision.utils import save_image
from torchvision.utils import make_grid
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

sns.set(style='darkgrid', font_scale=1.2)

## Часть 1. Подготовка данных (0.5 балла)

В качестве обучающей выборки возьмем часть датасета [Flickr Faces](https://github.com/NVlabs/ffhq-dataset), который содержит изображения лиц людей в высоком разрешении (1024х1024). Оригинальный датасет очень большой, поэтому мы возьмем его часть. Скачать датасет можно [здесь](https://www.kaggle.com/datasets/tommykamaz/faces-dataset-small?resource=download-directory) и  [здесь](https://drive.google.com/file/d/1inyvLrN5wKBGCxQ4znMKBc64uL4uP_2x/view?usp=drive_link)

Давайте загрузим наши изображения. Напишите функцию, которая строит DataLoader для изображений, при этом меняя их размер до нужного значения (размер 1024 слишком большой, поэтому мы рекомендуем взять размер 128 либо немного больше)

In [ ]:
import kagglehub

images_path = kagglehub.dataset_download("tommykamaz/faces-dataset-small")

In [ ]:
from torch.utils.data import DataLoader
from torchvision.datasets import ImageFolder
from torchvision import transforms as T


def get_dataloader(image_size, batch_size):
    """
    Builds dataloader for training data.
    Use tt.Compose and tt.Resize for transformations
    :param image_size: height and wdith of the image
    :param batch_size: batch_size of the dataloader
    :returns: DataLoader object
    """

    transform = T.Compose([
        T.Resize(image_size),
        T.ToTensor(),
        T.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
    ])

    dataset = ImageFolder(root=images_path, transform=transform)
    dataloader = DataLoader(dataset, batch_size=batch_size,
                            shuffle=True, num_workers=8)
    
    return dataloader

In [ ]:
image_size = 128
batch_size = 64
device = 'mps'

dataloader = get_dataloader(image_size, batch_size)

In [ ]:
batch = next(iter(dataloader))
images, _ = batch
images = images * 0.5 + 0.5
grid = make_grid(images[:16], nrow=4)

plt.figure(figsize=(8, 8))
plt.axis("off")
plt.title("Примеры изображений из обучающей выборки")
plt.imshow(grid.permute(1, 2, 0))
plt.show()

## Часть 2. Построение и обучение модели (2 балла)

Сконструируйте генератор и дискриминатор. Помните, что:
* дискриминатор принимает на вход изображение (тензор размера `3 x image_size x image_size`) и выдает вероятность того, что изображение настоящее (тензор размера 1)

* генератор принимает на вход тензор шумов размера `latent_size x 1 x 1` и генерирует изображение размера `3 x image_size x image_size`

In [ ]:
class Generator(nn.Module):
    def __init__(self, latent_size): #image_size=128
        super().__init__()
        self.net = nn.Sequential(
            nn.ConvTranspose2d(latent_size, 1024, 4, 1, 0, bias=False),
            nn.BatchNorm2d(1024),
            nn.ReLU(True),

            nn.ConvTranspose2d(1024, 512, 4, 2, 1, bias=False),
            nn.BatchNorm2d(512),
            nn.ReLU(True),

            nn.ConvTranspose2d(512, 256, 4, 2, 1, bias=False),
            nn.BatchNorm2d(256),
            nn.ReLU(True),

            nn.ConvTranspose2d(256, 128, 4, 2, 1, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(True),

            nn.ConvTranspose2d(128, 64, 4, 2, 1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(True),

            nn.ConvTranspose2d(64, 3, 4, 2, 1, bias=False),
            nn.Tanh()
        )

    def forward(self, x):
        return self.net(x)


class Discriminator(nn.Module):
    def __init__(self): #image_size=128
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=4, stride=2, padding=1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(64, 128, kernel_size=4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Dropout(0.3),

            nn.Conv2d(128, 256, kernel_size=4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(256),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Dropout(0.3),

            nn.Conv2d(256, 512, kernel_size=4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(512),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(512, 1024, 4, 2, 1, bias=False),
            nn.BatchNorm2d(1024),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(1024, 1, 4, 1, 0, bias=False),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.net(x).view(x.size(0))

In [ ]:
latent_size = 256

generator = Generator(latent_size)
discriminator = Discriminator()

Перейдем теперь к обучению нашего GANа. Алгоритм обучения следующий:
1. Учим дискриминатор:
  * берем реальные изображения и присваиваем им метку 1
  * генерируем изображения генератором и присваиваем им метку 0
  * обучаем классификатор на два класса

2. Учим генератор:
  * генерируем изображения генератором и присваиваем им метку 0
  * предсказываем дискриминаторором, реальное это изображение или нет


В качестве функции потерь берем бинарную кросс-энтропию

In [ ]:
model = {
    "discriminator": discriminator.to(device),
    "generator": generator.to(device)
}

criterion = {
    "discriminator": nn.BCELoss().to(device),
    "generator": nn.BCELoss().to(device)
}

In [ ]:
import torch.optim as optim

def fit(model, criterion, dataloader, epochs, lr, device,
        sample_every=10, display_images=True):
    
    optimizer = {
        "generator": optim.Adam(model["generator"].parameters(),
                                lr=lr, betas=(0.5, 0.999)),
        "discriminator": optim.Adam(model["discriminator"].parameters(),
                                    lr=lr, betas=(0.5, 0.999))
    }

    scheduler = {
        "generator": optim.lr_scheduler.StepLR(
            optimizer["generator"],
            step_size=30, gamma=0.7
        ),
        "discriminator": optim.lr_scheduler.StepLR(
            optimizer["discriminator"],
            step_size=30, gamma=0.7
        )
    }

    loss = {
        "generator": [],
        "discriminator": []
    }

    fixed_noise = torch.randn(64, latent_size, 1, 1, device=device)

    for epoch in tqdm(range(epochs), desc="Epochs"):
        model["generator"].train()
        model["discriminator"].train()

        total = {"generator": 0.0, "discriminator": 0.0}

        for real_images, _ in tqdm(dataloader,
                                   desc=f"Epoch {epoch+1}/{epochs}",
                                   leave=True):
            batch_size = real_images.size(0)
            real_images = real_images.to(device)
            real_labels = torch.ones(batch_size, device=device)
            fake_labels = torch.zeros(batch_size, device=device)

            # Discriminator
            z = torch.randn(batch_size, latent_size, 1, 1, device=device)
            fake_images = model["generator"](z).detach()

            out_real = model["discriminator"](real_images).view(-1)
            out_fake = model["discriminator"](fake_images).view(-1)

            loss_real = criterion["discriminator"](out_real, real_labels)
            loss_fake = criterion["discriminator"](out_fake, fake_labels)
            loss_D = loss_real + loss_fake

            optimizer["discriminator"].zero_grad()
            loss_D.backward()
            optimizer["discriminator"].step()

            total["discriminator"] += loss_D.item()

            # Generator
            z = torch.randn(batch_size, latent_size, 1, 1, device=device)
            fake_images = model["generator"](z)
            out = model["discriminator"](fake_images).view(-1)
            loss_G = criterion["generator"](out, real_labels)

            optimizer["generator"].zero_grad()
            loss_G.backward()
            optimizer["generator"].step()

            total["generator"] += loss_G.item()

        avg_G = total["generator"] / len(dataloader)
        avg_D = total["discriminator"] / len(dataloader)
        loss["generator"].append(avg_G)
        loss["discriminator"].append(avg_D)

        scheduler["generator"].step()
        scheduler["discriminator"].step()

        print(f"[{epoch+1}/{epochs}] Loss D: {avg_D:.4f}, Loss G: {avg_G:.4f};",
              f"lr D: {optimizer['discriminator'].param_groups[0]['lr']:.2e},",
              f"lr G: {optimizer['generator'].param_groups[0]['lr']:.2e}")

        if display_images and ((epoch + 1) % sample_every == 0 or epoch == 0):
            model["generator"].eval()
            with torch.no_grad():
                fake_samples = model["generator"](fixed_noise).detach().cpu()
                fake_samples = fake_samples * 0.5 + 0.5  # [-1,1] -> [0,1]
                grid = make_grid(fake_samples, nrow=8)

                plt.figure(figsize=(6, 6))
                plt.imshow(grid.permute(1, 2, 0).numpy())
                plt.title(f"Generated Samples at Epoch {epoch+1}")
                plt.axis("off")
                plt.show()

    plt.figure(figsize=(10, 5))
    plt.plot(loss["generator"], label="Generator")
    plt.plot(loss["discriminator"], label="Discriminator")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title("GAN Training Losses")
    plt.legend()
    plt.grid(True)
    plt.show()

    return model, loss

In [ ]:
model, loss = fit(
    model, criterion, dataloader,
    epochs=150,
    lr=0.0001,
    device='mps',
    sample_every=1,
)

In [ ]:
torch.save(model["generator"].state_dict(), "generator.pth")
torch.save(model["discriminator"].state_dict(), "discriminator.pth")

Постройте графики лосса для генератора и дискриминатора. Что вы можете сказать про эти графики?

**См. график выше.**
Удалось добиться каких-никаких человеческих результатов. Проблема здесь похожа на проблему, с которой столкнулись в задании по автоэнкодерам: вообще говоря, не гарантируется, что случайный вектор $z$ попадёт в «обученную» область латентного пространства генератора

Видно, что генератор отчаянно пытался попасть в моду дискриминатора, поэтому появилось много радостных, улыбающихся лиц. Будем считать это позитивным знаком! :)

## Часть 3. Генерация изображений

Теперь давайте оценим качество получившихся изображений. Напишите функцию, которая выводит изображения, сгенерированные нашим генератором

Как вам качество получившихся изображений?

**Я это уже сделал выше (в ходе обучения).** Но у вас могут быть "спрятаны" эти ячейки по умолчанию. Конечно, можно развернуть и поскроллить, но давайте повторим для удобства, заодно подкрутим масштаб:

In [ ]:
for model_name in model.keys():
    model[model_name].load_state_dict(
        torch.load(model_name + ".pth", weights_only=True)
    )

In [ ]:
fixed_noise = torch.randn(64, latent_size, 1, 1, device=device)

fake_samples = model["generator"](fixed_noise).detach().cpu()
fake_samples = fake_samples * 0.5 + 0.5  # [-1,1] -> [0,1]
grid = make_grid(fake_samples, nrow=8)

plt.figure(figsize=(12, 12))
plt.imshow(grid.permute(1, 2, 0).numpy())
plt.title(f"Generated Samples")
plt.axis("off")
plt.show()

**Вывод.** Качество ожидаемо не очень, поскольку сетка довольно примитивная.
Лица очень похожи на то, что получалось (со схожей архитектурой) в задании по автоэнкодерам (думаю, здесь параллель довольно очевидная) в части генерации случайных лиц с VAE.
Это логично: по сути генератор подбирает распределение в некотором пространстве, реализация выборки из которого «удовлетворяет» дискриминатор (соответственно, результат поход на распределение по реальным лицам из обучающей выборки).

## Часть 4. Leave-one-out-1-NN classifier accuracy (2.5 балла)

### 4.1. Подсчет accuracy (1.5 балл)

Не всегда бывает удобно оценивать качество сгенерированных картинок глазами. В качестве альтернативы вам предлагается реализовать следующий подход:
  * Сгенерировать столько же фейковых изображений, сколько есть настоящих в обучающей выборке. Присвоить фейковым метку класса 0, настоящим – 1.
  * Построить leave-one-out оценку: обучить 1NN Classifier (`sklearn.neighbors.KNeighborsClassifier(n_neighbors=1)`) предсказывать класс на всех объектах, кроме одного, проверить качество (accuracy) на оставшемся объекте. В этом вам поможет `sklearn.model_selection.LeaveOneOut`

**Есть понимание,** что обучать KNN на сырых картинках 128x128x3 = 49k не особо осмысленно.  
Такие данные высокоразмерные, шумные... сами всё понимаете.
Будем использовать фичи из ResNet18

In [ ]:
from torchvision.models import resnet18

resnet = resnet18(weights=True)
resnet.fc = nn.Identity()
resnet.eval()
resnet.to(device)

In [ ]:
@torch.no_grad()
def extract_features(images, model):
    images = images.to(device)
    features = model(images)
    return features.cpu()

In [ ]:
resnet_transform = T.Compose([
    T.Resize(224),
    T.Normalize(mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225])
    # как в ImageNet
])

real_images, _ = next(iter(dataloader))
real_images = real_images.to(device)
real_images = resnet_transform(real_images)

latent = torch.randn(real_images.size(0), latent_size, 1, 1, device=device)
fake_images = generator(latent).detach()
fake_images = resnet_transform(fake_images)

In [ ]:
real_features = extract_features(real_images, resnet)
fake_features = extract_features(fake_images, resnet)

X = torch.cat([real_features, fake_features], dim=0).numpy()
y = [1] * len(real_features) + [0] * len(fake_features)

In [ ]:
from sklearn.model_selection import LeaveOneOut
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

loo = LeaveOneOut()
clf = KNeighborsClassifier(n_neighbors=1)

y_true, y_pred = [], []

for train_idx, test_idx in tqdm(loo.split(X)):
    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = [y[i] for i in train_idx], [y[i] for i in test_idx]
    
    clf.fit(X_train, y_train)
    y_pred.append(clf.predict(X_test)[0])
    y_true.append(y_test[0])

accuracy = accuracy_score(y_true, y_pred)
print(f"Leave-one-out 1NN accuracy: {accuracy:.4f}")

Что вы можете сказать о получившемся результате? Какой accuracy мы хотели бы получить и почему?

**Accuracy** характеризует «противоборство» генератора и дискриминатора:
- если accuracy << 0.5 — дискриминатор слаб или переобучился, поскольку принимает фейки за настоящие изображения или наоборот
- если accuracy >> 0.5 — генератор слаб (реальные картинки и фейки легко отличимы)

Полученный результат 0.75 означает, что 75 % объектов правильно классифицируются по признакам ResNet18 как реальные или фейки. Значит, различие между картинками всё-таки есть, но в целом генератор попадает в общее распределение.

Идеальная генерация давала бы **accuracy &approx; 0.5**, то есть случайную угадайку.

### 4.2. Визуализация распределений (1 балл)

Давайте посмотрим на то, насколько похожи распределения настоящих и фейковых изображений. Для этого воспользуйтесь методом, снижающим размерность (к примеру, TSNE) и изобразите на графике разным цветом точки, соответствующие реальным и сгенерированным изображенияи

In [ ]:
from sklearn.manifold import TSNE

# Применяем t-SNE
tsne = TSNE(n_components=2, perplexity=30, max_iter=1000, random_state=42)
X_tsne = tsne.fit_transform(X)

plt.figure(figsize=(8, 6))
sns.scatterplot(x=X_tsne[:, 0], y=X_tsne[:, 1], hue=y, palette=['red', 'blue'], alpha=0.7)
plt.title("t-SNE распределение: реальные (1) vs фейковые (0)")
plt.xlabel("t-SNE dim 1")
plt.ylabel("t-SNE dim 2")
plt.legend(title='Класс')
plt.grid(True)
plt.show()

Прокомментируйте получившийся результат:

**Во-первых, это красиво!**
Видимо, что генератор пытался воспринять форму распределения, но лёг «в пузырь» (так что «реальные» точки «окружили» фейков): фейки  формируют компактную внутреннюю область. Реальные изображения более рассеяны и, похоже, охватывают большее пространство в признаковом слое ResNet18

Вероятно, генератор воспроизводит ограниченное подмножество признаков, распознаваемых дискриминатором как правдоподобные (коллапс моды). В любом случае — покрытие латентного пространства неполное. Генератору удалось приблизиться к реальному распределению, но ещё есть куда улучшаться... StyleGAN ждёт нас!